# Daily Sales Time-Series Analysis

A comprehensive exploration of daily sales patterns, trends, seasonality, holiday effects, and structural breaks to support demand planning and forecasting decisions.

## Project Overview

This notebook analyzes three years of synthetic daily sales data for a retail business. We decompose the series into trend, weekly seasonality, monthly cycles, and holiday spikes. We compute rolling averages, identify structural breaks, and derive actionable insights for inventory and staffing decisions.

## Learning Objectives

- Understand how to build and inspect a time-series DataFrame with realistic components
- Compute and interpret 7-day and 30-day rolling averages
- Identify weekly and monthly seasonality patterns
- Quantify holiday lift over baseline sales
- Discuss structural breaks and their business causes
- Prepare the data for downstream forecasting models

## Business or Research Problem

A retail chain wants to understand its daily sales dynamics before building a forecasting model. The key questions are:
- Is there a long-term growth trend?
- Which days of the week perform best and worst?
- How large is the Q4 holiday spike?
- Are there structural breaks (e.g., store expansions, promotions) in the series?

## Why This Analysis Matters

Accurate understanding of sales patterns directly impacts:
- **Inventory management**: Avoid overstocking or stockouts around peak periods
- **Staffing**: Schedule more staff on high-demand days
- **Cash flow forecasting**: Prepare for revenue troughs and spikes
- **Marketing timing**: Launch campaigns before predicted demand surges

This exploratory analysis is the essential first step before any forecasting pipeline.

## Dataset Overview

The synthetic dataset covers **1,095 days** (3 years: 2021–2023). It models:

| Column | Description |
|---|---|
| `date` | Calendar date |
| `sales` | Daily sales revenue (USD) |
| `day_of_week` | 0=Monday … 6=Sunday |
| `month` | 1–12 |
| `year` | 2021, 2022, 2023 |
| `is_holiday` | 1 if a major holiday window |
| `rolling_7` | 7-day centered rolling mean |
| `rolling_30` | 30-day rolling mean |

Components baked in: upward linear trend, weekly seasonality (weekends ~20% lower), monthly cycles, Q4 holiday spikes (+40%), and Gaussian noise.

## Dataset Source and License Notes

This dataset is **fully synthetic** and generated within this notebook using NumPy and Pandas. No external files or APIs are required. All values are for educational purposes only and do not represent any real business. You may freely reproduce and modify this notebook.

## Environment Setup

We use standard Python data science libraries. Install any missing ones with:

```bash
pip install pandas numpy matplotlib seaborn scipy
```

All libraries are available in any standard Anaconda or pip environment.

## Imports

Load all required libraries before any analysis.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='tab10')
print('Libraries loaded successfully.')

## Configuration / Constants

Centralize all tunable parameters so the notebook is easy to modify.

In [ ]:
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

START_DATE = '2021-01-01'
END_DATE   = '2023-12-31'
BASE_SALES = 5000          # baseline daily sales in USD
TREND_SLOPE = 0.8          # daily linear growth in USD
NOISE_STD = 400            # standard deviation of Gaussian noise
WEEKEND_DISCOUNT = 0.20    # weekends 20% below trend
HOLIDAY_LIFT = 0.40        # Q4 holidays 40% above trend
ROLLING_SHORT = 7
ROLLING_LONG  = 30

print(f'Date range: {START_DATE} to {END_DATE}')
print(f'Base sales: ${BASE_SALES:,} | Trend: +${TREND_SLOPE}/day | Noise std: ${NOISE_STD}')

## Dataset Download or Loading

We generate the entire dataset synthetically. The sales signal is composed of:
1. A linear upward trend
2. Weekly seasonality (weekends lower)
3. Monthly sinusoidal cycle
4. Q4 holiday spike
5. Gaussian noise

This mimics a realistic retail time series without any external dependency.

In [ ]:
dates = pd.date_range(start=START_DATE, end=END_DATE, freq='D')
n = len(dates)
t = np.arange(n)

# 1. Linear trend
trend = BASE_SALES + TREND_SLOPE * t

# 2. Weekly seasonality: lower on Sat(5) and Sun(6)
dow = dates.dayofweek
weekly = np.where(dow >= 5, -WEEKEND_DISCOUNT * BASE_SALES, 0)

# 3. Monthly sinusoidal cycle (peak mid-month)
monthly = 300 * np.sin(2 * np.pi * dates.month / 12)

# 4. Q4 holiday spike: Nov 20 - Jan 5 window each year
def is_holiday(d):
    m, day = d.month, d.day
    return int((m == 11 and day >= 20) or m == 12 or (m == 1 and day <= 5))

holiday_flag = np.array([is_holiday(d) for d in dates])
holiday_effect = holiday_flag * HOLIDAY_LIFT * BASE_SALES

# 5. Noise
noise = np.random.normal(0, NOISE_STD, n)

sales = trend + weekly + monthly + holiday_effect + noise
sales = np.maximum(sales, 0)  # no negative sales

df = pd.DataFrame({
    'date': dates,
    'sales': sales.round(2),
    'day_of_week': dates.dayofweek,
    'month': dates.month,
    'year': dates.year,
    'is_holiday': holiday_flag
})
df.set_index('date', inplace=True)

# Rolling averages
df['rolling_7']  = df['sales'].rolling(ROLLING_SHORT, center=True).mean()
df['rolling_30'] = df['sales'].rolling(ROLLING_LONG).mean()

print(f'Dataset shape: {df.shape}')
df.head()

## Data Validation Checks

Before analyzing, verify data quality: no missing values, no negative sales, correct date range, and sensible distributions.

In [ ]:
print('=== Data Validation ===')
print(f'Shape            : {df.shape}')
print(f'Date range       : {df.index.min().date()} to {df.index.max().date()}')
print(f'Missing values   :\n{df.isnull().sum()}')
print(f'Negative sales   : {(df["sales"] < 0).sum()}')
print(f'Sales min/max    : ${df["sales"].min():,.2f} / ${df["sales"].max():,.2f}')
print(f'Holiday days     : {df["is_holiday"].sum()}')
assert df['sales'].min() >= 0, 'Negative sales detected!'
print('\nAll validation checks passed.')

## Data Cleaning

The synthetic data is clean by construction. In a real project, cleaning would involve handling missing dates, outlier capping, and timezone normalization. Here we document what would be done and confirm no issues exist.

In [ ]:
# Confirm continuous date index (no gaps)
expected_days = (df.index.max() - df.index.min()).days + 1
actual_days = len(df)
print(f'Expected days : {expected_days}')
print(f'Actual rows   : {actual_days}')
assert expected_days == actual_days, 'Date gaps detected!'

# Describe
df[['sales']].describe().round(2)

## Exploratory Data Analysis

We start with a high-level view of the full time series, overlaid with rolling averages, to detect the overall trend and volatility structure.

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(df.index, df['sales'], alpha=0.4, color='steelblue', linewidth=0.8, label='Daily Sales')
ax.plot(df.index, df['rolling_7'],  color='orange', linewidth=1.5, label='7-Day Rolling Mean')
ax.plot(df.index, df['rolling_30'], color='crimson', linewidth=2.0, label='30-Day Rolling Mean')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
plt.xticks(rotation=45)
ax.set_title('Daily Sales with Rolling Averages (2021–2023)', fontsize=14)
ax.set_xlabel('Date')
ax.set_ylabel('Sales (USD)')
ax.legend()
plt.tight_layout()
plt.show()

**Interpretation:** The raw daily series (blue) shows high day-to-day volatility. The 7-day rolling mean (orange) smooths the weekly pattern, revealing the underlying trend more clearly. The 30-day rolling mean (red) shows the long-run growth trajectory and the Q4 holiday spikes recurring each year.

## Univariate Analysis

Examine the distribution of daily sales values to understand spread, skewness, and outliers.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(df['sales'], bins=50, color='steelblue', edgecolor='white')
axes[0].axvline(df['sales'].mean(), color='red', linestyle='--', label=f'Mean: ${df["sales"].mean():,.0f}')
axes[0].axvline(df['sales'].median(), color='orange', linestyle='--', label=f'Median: ${df["sales"].median():,.0f}')
axes[0].set_title('Distribution of Daily Sales')
axes[0].set_xlabel('Sales (USD)')
axes[0].legend()

axes[1].boxplot(df['sales'], vert=True, patch_artist=True,
                boxprops=dict(facecolor='steelblue', alpha=0.6))
axes[1].set_title('Boxplot of Daily Sales')
axes[1].set_ylabel('Sales (USD)')

plt.tight_layout()
plt.show()

skew = df['sales'].skew()
print(f'Skewness: {skew:.3f}')
print(f'Mean: ${df["sales"].mean():,.2f}   Median: ${df["sales"].median():,.2f}')
print(f'Std Dev: ${df["sales"].std():,.2f}')

**Interpretation:** The distribution is roughly bell-shaped with a slight right skew due to Q4 holiday peaks. The mean and median are close, indicating no extreme distortion. The boxplot reveals some high-value outliers corresponding to peak holiday days.

## Bivariate / Multivariate Analysis

Explore how sales relate to year, month, day-of-week, and holiday status simultaneously.

In [ ]:
monthly_avg = df.groupby(['year', 'month'])['sales'].mean().reset_index()
monthly_pivot = monthly_avg.pivot(index='month', columns='year', values='sales')

fig, ax = plt.subplots(figsize=(12, 5))
monthly_pivot.plot(ax=ax, marker='o', linewidth=2)
ax.set_title('Average Monthly Sales by Year')
ax.set_xlabel('Month')
ax.set_ylabel('Avg Daily Sales (USD)')
ax.set_xticks(range(1, 13))
ax.set_xticklabels(['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec'])
ax.legend(title='Year')
plt.tight_layout()
plt.show()

**Interpretation:** Across all three years, December stands out as the peak month due to holiday effects. The year-over-year lines show an upward shift, confirming the underlying growth trend. The sinusoidal monthly cycle is also visible with a mid-year local peak.

## Feature-Specific Insights

Analyze day-of-week and holiday effects in detail.

In [ ]:
day_names = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']
dow_avg = df.groupby('day_of_week')['sales'].mean()

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].bar(day_names, dow_avg.values, color=sns.color_palette('tab10', 7))
axes[0].set_title('Average Sales by Day of Week')
axes[0].set_ylabel('Avg Sales (USD)')
for i, v in enumerate(dow_avg.values):
    axes[0].text(i, v + 50, f'${v:,.0f}', ha='center', fontsize=8)

holiday_comp = df.groupby('is_holiday')['sales'].mean()
axes[1].bar(['Non-Holiday', 'Holiday'], holiday_comp.values, color=['steelblue', 'tomato'])
axes[1].set_title('Average Sales: Holiday vs Non-Holiday')
axes[1].set_ylabel('Avg Sales (USD)')
lift = (holiday_comp[1] - holiday_comp[0]) / holiday_comp[0] * 100
axes[1].text(1, holiday_comp[1] + 50, f'+{lift:.1f}%', ha='center', color='tomato', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.show()

print(f'Weekday avg : ${dow_avg[:5].mean():,.2f}')
print(f'Weekend avg : ${dow_avg[5:].mean():,.2f}')
print(f'Holiday lift: +{lift:.1f}%')

**Interpretation:** Weekdays consistently outperform weekends by approximately 20%, reflecting the business model's reliance on working-day traffic. Holiday periods show a significant lift, confirming that Q4 promotional windows drive material revenue upside.

## Statistical Checks

We test for trend significance and check the autocorrelation structure to confirm that the series is not just random noise.

In [ ]:
# Mann-Kendall-style trend test using Pearson correlation with time index
t_idx = np.arange(len(df))
corr, pval = stats.pearsonr(t_idx, df['sales'])
print(f'Pearson correlation (time vs sales): r={corr:.4f}, p={pval:.4e}')
print(f'Conclusion: {"Significant upward trend" if pval < 0.05 else "No significant trend"}')

# Year-over-year growth
annual = df.groupby('year')['sales'].sum()
print('\nAnnual Total Sales:')
for yr, amt in annual.items():
    print(f'  {yr}: ${amt:,.0f}')
yoy_growth = (annual[2023] - annual[2021]) / annual[2021] * 100
print(f'\n3-Year growth: +{yoy_growth:.1f}%')

## Visual Analysis

Decomposition-style charts: trend, monthly aggregation heatmap, and year-over-year overlay.

In [ ]:
# Monthly total sales heatmap
monthly_total = df.groupby(['year', 'month'])['sales'].sum().reset_index()
pivot = monthly_total.pivot(index='year', columns='month', values='sales')
pivot.columns = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']

fig, ax = plt.subplots(figsize=(14, 3))
sns.heatmap(pivot, annot=True, fmt='.0f', cmap='YlOrRd', ax=ax, linewidths=0.5)
ax.set_title('Monthly Total Sales Heatmap (USD)', fontsize=13)
ax.set_ylabel('Year')
plt.tight_layout()
plt.show()

**Interpretation:** The heatmap confirms December as the peak month every year. January dips due to post-holiday normalization. The year-over-year growth is visible in the shifting color intensity from 2021 to 2023.

In [ ]:
# Structural break indicator: 30-day rolling std
df['rolling_std_30'] = df['sales'].rolling(30).std()

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(df.index, df['rolling_std_30'], color='purple', linewidth=1.2)
ax.set_title('30-Day Rolling Std Dev of Sales (Volatility Proxy)', fontsize=13)
ax.set_xlabel('Date')
ax.set_ylabel('Rolling Std Dev (USD)')
plt.tight_layout()
plt.show()

**Interpretation:** Spikes in rolling standard deviation indicate periods of elevated volatility — typically the holiday transition windows (late November and early January). These are potential structural break points where a forecasting model should account for regime changes.

## Key Findings

Summarize the quantitative results derived from the analysis above.

In [ ]:
mean_sales = df['sales'].mean()
peak_day = df['sales'].idxmax()
peak_val = df['sales'].max()
low_day  = df['sales'].idxmin()
low_val  = df['sales'].min()
best_dow = day_names[int(dow_avg.idxmax())]
worst_dow = day_names[int(dow_avg.idxmin())]

print('=== Key Findings ===')
print(f'1. Average daily sales       : ${mean_sales:,.2f}')
print(f'2. Peak sales day            : {peak_day.date()} (${peak_val:,.2f})')
print(f'3. Lowest sales day          : {low_day.date()} (${low_val:,.2f})')
print(f'4. Best day of week          : {best_dow} (${dow_avg.max():,.2f} avg)')
print(f'5. Worst day of week         : {worst_dow} (${dow_avg.min():,.2f} avg)')
print(f'6. Holiday lift              : +{lift:.1f}% vs non-holiday')
print(f'7. 3-year revenue growth     : +{yoy_growth:.1f}%')
print(f'8. Trend significance        : r={corr:.3f}, p={pval:.2e}')

## Limitations

1. **Synthetic data**: The patterns are engineered and may not reflect real business complexity (e.g., product mix, competitor actions, supply disruptions).
2. **No external regressors**: Real sales are influenced by pricing, promotions, marketing spend, and macroeconomic factors not captured here.
3. **Single series**: A real analysis would include multiple SKUs or store-level data requiring hierarchical modeling.
4. **Simplified seasonality**: Real seasonality may have more complex harmonics than the single sinusoidal cycle modeled here.
5. **No causal inference**: Correlation between holiday flag and sales lift does not prove causality without A/B testing.

## Recommendations / Next Steps

1. **Build a forecasting model**: Use SARIMA, Prophet, or LightGBM with lag features to forecast the next 90 days.
2. **Add external regressors**: Include weather data, marketing spend, and economic indicators.
3. **Segment by product category**: Different categories may have distinct seasonal profiles.
4. **Automate holiday calendar**: Use a proper holiday library (e.g., `holidays` package) for country-specific calendars.
5. **Anomaly detection**: Flag days where sales deviate >2 standard deviations from the rolling mean for root-cause analysis.
6. **Structural break testing**: Apply the Chow test or PELT algorithm to formally detect regime changes.

## Common Mistakes

1. **Not checking for date gaps** before computing rolling averages — gaps cause incorrect window calculations.
2. **Using future data in rolling windows** (right-aligned vs. centered) for forecasting — always use trailing windows in production.
3. **Ignoring calendar effects** when splitting train/test data — always split on a calendar boundary.
4. **Confusing trend with seasonality** — a rising Q4 is partly seasonal, not purely growth.
5. **Over-smoothing with long rolling windows** — a 90-day mean may hide meaningful structural breaks.

## Mini Challenge / Exercises

1. **Modify the trend slope** to 2.0 and observe how the 3-year growth rate changes.
2. **Add a one-time structural break** (e.g., a store opening in July 2022 that adds +$1,000/day) and re-plot the rolling volatility.
3. **Compute week-of-year average sales** and plot a 52-week seasonality chart.
4. **Replicate the holiday lift analysis** separately for each year — is the lift growing or shrinking?
5. **Build a simple naive forecast**: predict tomorrow's sales as the 7-day rolling average and compute MAE on the last 90 days.

## Final Summary / Key Takeaways

- The dataset contains **1,095 days** of daily sales with an engineered upward trend, weekly seasonality, monthly cycles, and Q4 holiday spikes.
- **Rolling averages** are essential for smoothing noise and revealing the true underlying trend.
- **Weekdays outperform weekends** by ~20%, and **holiday periods deliver a significant sales lift** over baseline.
- **Year-over-year growth** is statistically significant (Pearson correlation p < 0.05), confirming a real trend.
- **December** is consistently the highest-revenue month; January shows the predictable post-holiday dip.
- The **30-day rolling standard deviation** is a useful proxy for identifying structural break windows.
- This exploratory analysis is the **necessary foundation** before building any forecasting model — understanding the data's structure prevents common modeling errors.